# Practice Lab: Memory Systems for Agents (Lesson 11.4)

Module 11 · 8 exercises · Buffer + summarization + vector + episodic + CLAUDE.md + graphs + DPDP + India

Exercises 1-4 run locally (pure Python).


# Lesson 11.4: Memory Systems for Agents

8 exercises: 2E + 3M + 3C

Exercises 1-4 run **locally** (pure Python). Ex 5-8 are design.


In [ ]:
import numpy as np
import json
from datetime import datetime, timedelta
np.random.seed(42)


---
## Exercise 1 (Easy): U-Curve Tester


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np
np.random.seed(42)

def sim_ucurve(n=20, trials=100):
    accs = []
    for pos in range(n):
        norm = pos/(n-1)
        u = 4*(norm-0.5)**2
        base = 0.65+0.30*u
        accs.append(float(np.mean(np.clip(np.random.normal(base,0.05,trials),0,1))))
    return accs

print("U-Curve Tester:")
accs = sim_ucurve(20)
for zone,sl in [("Start",slice(0,4)),("Middle",slice(8,12)),("End",slice(16,20))]:
    avg = np.mean(accs[sl])
    print(f"  {zone:<8} {avg:.0%}  {'SAFE' if avg>0.85 else 'DANGER'}")
print(f"\n  Recommendations:")
print(f"  Start: system prompt. End: current task. Middle: history.")
print(f"  Effective context = ~65-70% of advertised")
print(f"  Compact at 70-85% utilization")


</details>


---
## Exercise 2 (Easy): CLAUDE.md Creator


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
print("CLAUDE.md Creator:")
hierarchy = [
    ("~/.claude/CLAUDE.md","Global","No","Never use em dashes; Python 3.13+; uv; INR/IST"),
    ("./CLAUDE.md","Project (git-shared)","Yes","Netsetos: dark theme, emerald #059669, 4-5 quizzes/lesson"),
    ("./CLAUDE.local.md","Personal","No (gitignored)","My API keys, working branch, skip exercises"),
    ("./modules/m9/CLAUDE.md","Subdirectory","Yes","Module 11: emerald, simulated tools, no API calls"),
]
print(f"\n  Loading Order:")
for i,(path,scope,git,content) in enumerate(hierarchy,1):
    print(f"  {i}. {path}")
    print(f"     {scope} | git={git}")
    print(f"     Content: {content}")

print(f"\n  CLAUDE.md = YOUR instructions (manual)")
print(f"  MEMORY.md = CLAUDE'S notebook (auto at ~/.claude/projects/)")
print(f"  ETH Zurich: LLM-generated context REDUCED success ~3%")
print(f"  Include ONLY what agents cannot discover independently")


</details>


---
## Exercise 3 (Medium): Mem0 Integration


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np
np.random.seed(42)

class SimMem0:
    def __init__(self): self.memories=[]; self.total_tok=0
    def add(self, msgs):
        self.total_tok += sum(len(m.split())*1.3 for m in msgs)
        for m in msgs:
            w = m.lower()
            if any(k in w for k in ["name","i'm"]): self.memories.append({"type":"id","text":m[:50],"tok":len(m.split())})
            elif any(k in w for k in ["budget","price","rupee"]): self.memories.append({"type":"pref","text":m[:50],"tok":len(m.split())})
            elif any(k in w for k in ["yesterday","last week"]): self.memories.append({"type":"temporal","text":m[:50],"tok":len(m.split())})
    def stats(self):
        mt=sum(m["tok"] for m in self.memories)
        return {"n":len(self.memories),"mem_tok":int(mt),"full_tok":int(self.total_tok),"savings":1-mt/max(self.total_tok,1)}

mem = SimMem0()
for sess in [["My name is Priya from Hyderabad","My budget is 15000 rupees"],
             ["I enrolled in GenAI yesterday","The attention section was confusing"],
             ["Last week I completed module 2","I want to focus on agents"]]:
    mem.add(sess)
s = mem.stats()
print(f"Mem0 Integration:")
print(f"  Memories: {s['n']}, Tokens: {s['mem_tok']} vs {s['full_tok']} full ({s['savings']:.0%} savings)")

print(f"\n  Accuracy Comparison:")
print(f"  {'Query':<28} {'Full':>6} {'Mem0':>6}")
for q,fa,ma in [("What's her name?",0.95,0.92),("Budget?",0.90,0.88),
                ("When did she enroll?",0.85,0.49),("Last week activity?",0.82,0.45)]:
    print(f"  {q:<28} {fa:>5.0%} {ma:>5.0%}")
print(f"\n  Mem0 wins: identity/prefs. Loses: temporal (use Zep)")


</details>


---
## Exercise 4 (Medium): Tiered Compaction


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class Compactor:
    def __init__(self,max_tok=500): self.msgs=[]; self.summary=""; self.max=max_tok
    def add(self,role,content,tool=False):
        self.msgs.append({"role":role,"content":content,"tool":tool,"tok":int(len(content.split())*1.3)})
    def tokens(self): return sum(m["tok"] for m in self.msgs)+(int(len(self.summary.split())*1.3) if self.summary else 0)
    def util(self): return self.tokens()/self.max

    def tier1(self):
        b=self.tokens(); n=len(self.msgs)
        self.msgs=[m for m in self.msgs if not m["tool"] or m in self.msgs[-4:]]
        return {"tier":1,"saved":b-self.tokens(),"removed":n-len(self.msgs),"retention":"100%"}

    def tier2(self):
        b=self.tokens()
        if len(self.msgs)>6:
            self.summary=f"Summary: course inquiry, {len(self.msgs)-4} turns compressed"
            self.msgs=self.msgs[-4:]
        return {"tier":2,"saved":b-self.tokens(),"retention":"~85%"}

    def tier3(self):
        b=self.tokens()
        self.summary="Compressed: course inquiry, pricing discussed"
        self.msgs=self.msgs[-2:]
        return {"tier":3,"saved":b-self.tokens(),"retention":"~60%"}

c = Compactor(500)
for r,t,tool in [("user","My name is Priya, I want GenAI course",False),
    ("assistant","GenAI course is 14999 INR, 146 hours",False),
    ("tool","search('genai') -> {price:14999}",True),
    ("user","What about Python course?",False),
    ("tool","search('python') -> {price:9999}",True),
    ("assistant","Python course is 9999 INR, 80 hours",False),
    ("user","Compare both with GST",False),
    ("tool","calc('14999*1.18') -> 17698.82",True),
    ("tool","calc('9999*1.18') -> 11798.82",True),
    ("assistant","GenAI: 17699, Python: 11799 INR",False),
    ("user","Which do you recommend?",False),
    ("assistant","Given your interests, GenAI is ideal",False)]:
    c.add(r,t,tool)

print(f"Tiered Compaction:")
print(f"  Initial: {len(c.msgs)} msgs, {c.tokens()} tok, {c.util():.0%} util")
for tier_fn,name in [(c.tier1,"MicroCompact"),(c.tier2,"AutoCompact"),(c.tier3,"FullCompact")]:
    r = tier_fn()
    print(f"  Tier {r['tier']} ({name}): saved={r['saved']} tok, retention={r['retention']}, now {c.util():.0%}")
print(f"\n  Pattern: lossless first, lossy only when needed")
print(f"  Claude Code: compact at 92%. Carnegie Mellon: 23% degradation >85%")


</details>


---
## Exercise 5 (Medium): LangGraph Store API
Design/architecture. See practice-lab-11_4.html.


In [ ]:
# Exercise 5: LangGraph Store API
pass


<details><summary>Solution</summary>


In [ ]:
print('LangGraph Store API: Checkpointer (within-thread) + Store (cross-thread)')
print('Checkpointer: auto at superstep, thread_id scoped, time-travel')
print('Store: namespace isolation, cross-session, store.put/get/search')
print('5 patterns: dedicated node, memory-as-tools, shared state, background, external')


</details>


---
## Exercise 6 (Challenge): Knowledge Graph Memory
Design/architecture. See practice-lab-11_4.html.


In [ ]:
# Exercise 6: Knowledge Graph Memory
pass


<details><summary>Solution</summary>


In [ ]:
print('Knowledge Graph: entity extraction (spaCy+LLM) -> Neo4j/FalkorDB')
print('Graph: 92% recall/88% precision vs vector 85%/75%')
print('Multi-hop: follows relationships (Priya->TechCorp->Netsetos->GenAI)')
print('LazyGraphRAG: 99% cheaper indexing ($0.50 vs $50-200). Start vectors, add graphs.')


</details>


---
## Exercise 7 (Challenge): DPDP-Compliant Memory
Design/architecture. See practice-lab-11_4.html.


In [ ]:
# Exercise 7: DPDP-Compliant Memory
pass


<details><summary>Solution</summary>


In [ ]:
from datetime import datetime, timedelta
print('DPDP-Compliant Memory: consent + 7-day erasure + audit')
print(f'Erasure deadline: {(datetime.now()+timedelta(days=7)).strftime("%Y-%m-%d")}')
print('Vector deletion gap: no provable deletion for embeddings')
print('Workarounds: metadata delete, per-user namespace, TTL, re-embed')
print('Penalty: up to 250 Cr INR. Full compliance May 2027')


</details>


---
## Exercise 8 (Challenge): India Multilingual Memory
Design/architecture. See practice-lab-11_4.html.


In [ ]:
# Exercise 8: India Multilingual Memory
pass


<details><summary>Solution</summary>


In [ ]:
print('India Multilingual Memory: bge-multilingual-gemma2 (+35% recall)')
print('Cross-language: store Hindi, retrieve English (88% recall, -7%)')
print('MuRIL: 0.99+ cosine sim Hindi/English/Hinglish')
print('Indian cloud: E2E H100 249rs/hr (60% off), Yotta 65rs/hr (subsidized)')


</details>
